# RoBERTa (youtube-xlm-roberta-base-sentiment-multilingual) warmup steps tuning

Adjusting `warmup_steps` across all 4 languages (English/Tamil/Hinglish/Korean), a baseline plus 3 warmup variants.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers accelerate --quiet

import os
import pandas as pd

TRAIN_DIR = '/content/drive/MyDrive/IAT 360 Final Project/Train'
TEST_DIR = '/content/drive/MyDrive/IAT 360 Final Project/Test'

LANGUAGE_FILES = {
    'english': ('df_english_train.csv', 'df_english_test.csv'),
    'tamil': ('df_tamil_train.csv', 'df_tamil_test.csv'),
    'hinglish': ('df_hinglish_train.csv', 'df_hinglish_test.csv'),
    'korean': ('df_korean_train.csv', 'df_korean_test.csv'),
}

def load_and_tag(dir_path, filename, language):
    df = pd.read_csv(os.path.join(dir_path, filename), engine='python', on_bad_lines='skip')
    df['Language'] = language
    df['Sentiment'] = df['Sentiment'].str.strip().str.capitalize()
    return df

train_dfs, test_dfs = [], []
for language, (train_file, test_file) in LANGUAGE_FILES.items():
    train_dfs.append(load_and_tag(TRAIN_DIR, train_file, language))
    test_dfs.append(load_and_tag(TEST_DIR, test_file, language))

df_train_all = pd.concat(train_dfs, ignore_index=True)
df_test_all = pd.concat(test_dfs, ignore_index=True)

In [ ]:
# cap train size per language for speed, keep full test set
def cap_per_language(df, n_per_language, seed=42):
    if n_per_language is None:
        return df
    parts = []
    for language in df['Language'].unique():
        subset = df[df['Language'] == language]
        parts.append(subset.sample(n=min(len(subset), n_per_language), random_state=seed))
    return pd.concat(parts, ignore_index=True)

train_split_df = cap_per_language(df_train_all, 8000)
df_test_all = cap_per_language(df_test_all, None)

print("Training on:", train_split_df.shape)
print("Evaluating on:", df_test_all.shape)

In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset

class SentimentDataset(TorchDataset):
    def __init__(self, dataframe, tokenizer, label2id, max_length=128):
        self.texts = dataframe['CommentText'].astype(str).tolist()
        self.labels = dataframe['Sentiment'].map(label2id).astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def data_collator(features):
    batch = {}
    for key in features[0].keys():
        stacked = torch.stack([f[key] for f in features])
        batch[key] = stacked.long() if key == 'labels' else stacked
    return batch

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score
from tqdm.auto import tqdm

LABEL_ORDER = [
    'English - Positive', 'English - Negative', 'English - Neutral',
    'Tamil - Positive', 'Tamil - Negative', 'Tamil - Neutral',
    'Hinglish - Positive', 'Hinglish - Negative', 'Hinglish - Neutral',
    'Korean - Positive', 'Korean - Negative', 'Korean - Neutral',
]

@torch.no_grad()
def predict_batch(model, tokenizer, id2label, comments, batch_size=64, max_length=128):
    device = model.device
    model.eval()
    predictions = []
    for i in tqdm(range(0, len(comments), batch_size), desc="Inference"):
        batch = [str(c) for c in comments[i:i + batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
        predictions.extend(id2label[p] for p in preds)
    return predictions

# scores + saves a confusion matrix/classification report for one run
def evaluate_and_report(model, tokenizer, id2label, test_df, run_label, save_dir):
    df = test_df.copy()
    df['Predicted_Sentiment'] = predict_batch(model, tokenizer, id2label, df['CommentText'].tolist())
    df['True_Label'] = df['Language'].str.capitalize() + ' - ' + df['Sentiment']
    df['Predicted_Label'] = df['Language'].str.capitalize() + ' - ' + df['Predicted_Sentiment']

    cm = confusion_matrix(df['True_Label'], df['Predicted_Label'], labels=LABEL_ORDER)
    fig, ax = plt.subplots(figsize=(14, 12))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_ORDER)
    disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=60)
    ax.set_title(f'RoBERTa \u2014 {run_label}')
    plt.tight_layout()
    safe_name = run_label.replace(' ', '_').replace('=', '')
    plt.savefig(os.path.join(save_dir, f'confusion_matrix_{safe_name}.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n=== {run_label} ===")
    print(classification_report(df['True_Label'], df['Predicted_Label'], labels=LABEL_ORDER))

    acc = accuracy_score(df['True_Label'], df['Predicted_Label'])
    macro_f1 = f1_score(df['True_Label'], df['Predicted_Label'], average='macro')
    return {'run': run_label, 'accuracy': acc, 'macro_f1': macro_f1, 'predictions_df': df}

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"
SAVE_DIR = '/content/drive/MyDrive/IAT 360 Final Project/RoBERTa_Warmup_Results'
os.makedirs(SAVE_DIR, exist_ok=True)

roberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def fresh_model():
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
    m.to('cuda' if torch.cuda.is_available() else 'cpu')
    return m

_tmp_model = fresh_model()
id2label = {k: v.capitalize() for k, v in _tmp_model.config.id2label.items()}
label2id = {v: k for k, v in id2label.items()}

## Hyperparameter tuning: warmup_steps

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# learning_rate fixed (that's laraine's hyperparameter, not mine); bracketing 0
# (no warmup) against two longer warmups
FIXED_LEARNING_RATE = 1e-5
WARMUP_STEPS_TO_FINETUNE = [0, 100, 300]

results = [evaluate_and_report(_tmp_model, roberta_tokenizer, id2label, df_test_all, 'Baseline (original, no fine-tuning)', SAVE_DIR)]

for warmup_steps in WARMUP_STEPS_TO_FINETUNE:
    print(f"\n{'='*60}\nFine-tuning at warmup_steps={warmup_steps}\n{'='*60}")

    model = fresh_model()
    train_dataset = SentimentDataset(train_split_df, roberta_tokenizer, label2id)

    training_args = TrainingArguments(
        output_dir=f'./roberta_warmup_{warmup_steps}',
        learning_rate=FIXED_LEARNING_RATE,
        warmup_steps=warmup_steps,  # <-- hyperparameter being changed
        num_train_epochs=3,
        per_device_train_batch_size=32,
        save_strategy='epoch',
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to=[],
        seed=42,
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
    )
    trainer.train()

    results.append(evaluate_and_report(model, roberta_tokenizer, id2label, df_test_all, f'Fine-tuned (warmup_steps={warmup_steps})', SAVE_DIR))

In [ ]:
comparison_df = pd.DataFrame([{'Run': r['run'], 'Accuracy': r['accuracy'], 'Macro F1': r['macro_f1']} for r in results])
print(comparison_df)
comparison_df.to_csv(os.path.join(SAVE_DIR, 'warmup_steps_comparison.csv'), index=False)

fig, ax = plt.subplots(figsize=(8, 5))
comparison_df.set_index('Run')[['Accuracy', 'Macro F1']].plot(kind='bar', ax=ax)
ax.set_title('RoBERTa Performance Across Warmup Steps')
ax.set_ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'warmup_steps_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()